In [ ]:
# 04 — Neural Models: MLP and TabTransformer (Member 2)

import sys, json, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns

import config as C
from data_prep import load_prepared
from evaluation import evaluate, print_report
from features import NUMERIC, CATEGORICALS
from nn_models import MLP, TabTransformer, count_parameters, set_all_seeds
from nn_trainer import (build_categorical_encoder, cardinalities,
                        fit_continuous_scaler, make_tensors,
                        make_onehot_tensors, train_model, predict_proba)
from train_neural import (_mlp_packs, tune_mlp, train_mlp_final,
                          _tt_packs, tune_tabtransformer, train_tt_final)

sns.set_theme(style="whitegrid")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE} | torch {torch.__version__}")

Xtr, ytr, Xva, yva, Xte, yte, meta = load_prepared()

In [ ]:
## 1. Architecture — why the categorical features matter

# TabTransformer runs self-attention over embedded categorical columns;
# continuous features are concatenated after the transformer stack and passed to
# the MLP head.

# categorical indices → column embeddings (+ per-column embedding)
#                     → N × Transformer blocks (MHSA + feed-forward)
#                     → flattened contextual embeddings
# continuous features → LayerNorm
#                     → concatenate → MLP head → 3 logits

tt_packs, tt_scaler, tt_enc = _tt_packs(meta, ytr, yva, yte)
cards = cardinalities(tt_enc)

print("categorical cardinalities (levels + 1 OOV slot):")
for col, c in zip(CATEGORICALS, cards):
    print(f"  {col:<16} {c-1} levels -> {sorted(tt_enc[col])}")
print(f"\ncontinuous features: {len(NUMERIC)}")
print(f"attention operates over {len(CATEGORICALS)} tokens per row")

In [ ]:
## 2. MLP baseline

mlp_packs, mlp_scaler = _mlp_packs(Xtr, ytr, Xva, yva, Xte, yte)
mlp_params, mlp_val = tune_mlp(mlp_packs, C.N_TRIALS_MLP, C.PRIMARY_SEED)
print(f"best validation F1-macro = {mlp_val:.4f}")
print(json.dumps(mlp_params, indent=2))

In [ ]:
mlp_model, mlp_res, mlp_hist, mlp_secs, mlp_npar = train_mlp_final(
    mlp_packs, mlp_params, C.PRIMARY_SEED)

print(f"test F1-macro = {mlp_res['test']['f1_macro']:.4f}")
print(f"{mlp_npar:,} parameters | {mlp_secs:.1f}s")

proba = predict_proba(mlp_model, mlp_packs[2][0], mlp_packs[2][1], device=DEVICE)
print_report(yte, proba.argmax(1), "MLP — test set")

In [ ]:
## 3. TabTransformer

tt_params, tt_val = tune_tabtransformer(tt_packs, cards,
                                        C.N_TRIALS_TABTRANSFORMER,
                                        C.PRIMARY_SEED)
print(f"best validation F1-macro = {tt_val:.4f}")
print(json.dumps(tt_params, indent=2))

In [ ]:
tt_model, tt_res, tt_hist, tt_secs, tt_npar = train_tt_final(
    tt_packs, cards, tt_params, C.PRIMARY_SEED)

print(f"test F1-macro = {tt_res['test']['f1_macro']:.4f}")
print(f"{tt_npar:,} parameters | {tt_secs:.1f}s")

proba_tt = predict_proba(tt_model, tt_packs[2][0], tt_packs[2][1], device=DEVICE)
print_report(yte, proba_tt.argmax(1), "TabTransformer — test set")

In [ ]:
## 4. Training curves

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h, col in [("MLP", mlp_hist, "#e9c46a"),
                     ("TabTransformer", tt_hist, "#e76f51")]:
    axes[0].plot(h.epoch, h.train_loss, color=col, label=f"{name} train", lw=1.2)
    axes[0].plot(h.epoch, h.val_loss, color=col, ls="--", label=f"{name} val", lw=1.2)
    axes[1].plot(h.epoch, h.val_f1_macro, color=col, label=name, lw=1.4)
    b = h.val_f1_macro.idxmax()
    axes[1].scatter(h.epoch[b], h.val_f1_macro[b], color=col, s=40, zorder=5)

axes[0].set(xlabel="Epoch", ylabel="Cross-entropy", title="Loss")
axes[1].set(xlabel="Epoch", ylabel="F1-macro",
            title="Validation F1-macro (dot = restored epoch)")

for a in axes: a.legend(fontsize=8)
plt.tight_layout()
plt.savefig(C.FIGURES_DIR / "nb04_training_curves.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
## 5. What the attention actually learned

tt_model.eval()
with torch.no_grad():
    maps = tt_model.attention_maps(tt_packs[2][0][:512].to(DEVICE),
                                   tt_packs[2][1][:512].to(DEVICE))

avg = np.mean([m.mean(axis=0) for m in maps], axis=0)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(avg, annot=True, fmt=".2f", cmap="Purples",
            xticklabels=CATEGORICALS, yticklabels=CATEGORICALS, ax=ax)
ax.set(title="Mean attention weights (all blocks)", xlabel="Key", ylabel="Query")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(C.FIGURES_DIR / "attention_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved -> results/figures/attention_heatmap.png")

print("Uniform ≈0.25 everywhere would mean attention learned nothing useful.")

In [ ]:
## 6. RQ1 verdict
# Reported as mean ± std across five seeds. A gap smaller than the pooled
# standard deviation is not evidence of anything.

import subprocess
subprocess.run([sys.executable, "../src/train_neural.py",
                "--trials", str(C.N_TRIALS_TABTRANSFORMER)], cwd="../src")

In [ ]:
summary = {}
for f in ["baseline_summary.json", "neural_summary.json"]:
    p = C.RESULTS_DIR / f
    if p.exists():
        summary.update(json.load(open(p)))

tbl = pd.DataFrame([
    {"model": k,
     "F1-macro": f"{v['f1_macro_mean']:.4f} ± {v['f1_macro_std']:.4f}",
     "Healthy F1": round(v["f1_healthy_mean"], 4),
     "AUC-ROC": round(v.get("auc_roc_macro_mean", float("nan")), 4),
     "train_s": round(v["train_seconds_mean"], 1),
     "params": int(v["n_params_mean"])}
    for k, v in summary.items()])
print(tbl.to_string(index=False))